In [1]:
from pathlib import Path

class CONLLUParser:
    def __init__(self):
        self.sentences = []
        self.texts = []

    def __make_wordform(self, parts):
        return {
            "id": int(parts[0]),
            "form": parts[1],
            "lemma": parts[2],
            "upos": parts[3],
            "head": int(parts[6]) if parts[6] != "_" else 0,
            "deprel": parts[7]
        }

    def __read_sentence(self, lines, pos):
        sent = []
        text = ""

        while pos < len(lines):
            line = lines[pos].rstrip("\n")

            if line == "":
                pos += 1
                break

            if line.startswith("# text = "):
                text = line[9:].strip()
            elif not line.startswith("#"):
                parts = line.split("\t")
                if len(parts) >= 8 and "-" not in parts[0] and "." not in parts[0]:
                    try:
                        sent.append(self.__make_wordform(parts))
                    except:
                        pass

            pos += 1

        return sent, text, pos

    def parse_conllu_file(self, filename):
        with open(filename, "r", encoding="utf-8") as f:
            lines = f.readlines()

        self.sentences = []
        self.texts = []

        pos = 0
        while pos < len(lines):
            sent, text, pos = self.__read_sentence(lines, pos)
            if sent:
                self.sentences.append(sent)
                self.texts.append(text)

    def find_construction(self, child_upos, child_deprel, parent_upos):
        results = []

        for sent_idx, sent in enumerate(self.sentences):
            id2word = {w["id"]: w for w in sent}

            for child in sent:
                if child["upos"] == child_upos and child["deprel"] == child_deprel:
                    parent = id2word.get(child["head"])
                    if parent and parent["upos"] == parent_upos:
                        results.append({
                            "text": self.texts[sent_idx],
                            "sentence": [w["form"] for w in sent],
                            "match": child,
                            "parent": parent
                        })
                        break

        return results


class ConstructionFinder:
    def __init__(self):
        self.parser = CONLLUParser()
        self.results = {}

    def find_in_directory(self, path, child_upos, child_deprel, parent_upos, filename_mask="*Gaydar*.conllu"):
        all_results = {}

        for file_path in Path(path).glob(filename_mask):
            self.parser.parse_conllu_file(str(file_path))
            found = self.parser.find_construction(child_upos, child_deprel, parent_upos)
            all_results[str(file_path)] = found
            self.results[str(file_path)] = found

        return all_results

In [6]:
from typing import Dict, List, Tuple

# Объявите ваш finder один раз
finder = ConstructionFinder()

# Список признаков: (parent_upos, direction, child_upos, deprel, label)
# direction: "<-" = parent ← child, "->" = parent → child
patterns = [
    ("VERB", "<-", "PART", "parataxis", "VERB, <-, PART, parataxis"),
    ("PRON", "<-", "ADP", "fixed", "PRON, <-, ADP, fixed"),
    ("PRON", "->", "NOUN", "nsubj", "PRON, ->, NOUN, nsubj"),
    ("NUM", "->", "NOUN", "flat", "NUM, ->, NOUN, flat"),
    ("PUNCT", "<-", "INTJ", "root", "PUNCT, <-, INTJ, root"),
    ("NOUN", "<-", "NOUN", "orphan", "NOUN, <-, NOUN, orphan"),
    ("PROPN", "<-", "NOUN", "nsubj", "PROPN, <-, NOUN, nsubj"),
    ("NOUN", "<-", "PROPN", "nmod", "NOUN, <-, PROPN, nmod"),
    ("ADJ", "->", "PRON", "nsubj", "ADJ, ->, PRON, nsubj"),
    ("VERB", "<-", "PART", "parataxis", "VERB, <-, PART, parataxis"),
    ("PRON", "<-", "VERB", "root", "PRON, <-, VERB, root"),
    ("ADJ", "<-", "NOUN", "nmod", "ADJ, <-, NOUN, nmod"),
    ("ADJ", "->", "PRON", "nsubj", "ADJ, ->, PRON, nsubj"),
    ("PAD", "<-", "VERB", "obl", "PAD, <-, VERB, obl"),
    ("PRON", "->", "AUX", "cop", "PRON, ->, AUX, cop")
]

In [7]:
# Для Гайдара — сначала пытаемся только Gaydar, потом все *.conllu
filename_masks = ["*Mamina*.conllu"]

for parent_upos, direction, child_upos, deprel, label in patterns:
    print(f"\n=== {label} ===")
    
    example_found = False
    
    # Попробовать сначала Gaydar, потом все файлы
    for mask in filename_masks:
        # Интерпретируем направление стрелки
        if direction == "<-":
            parent, child = child_upos, parent_upos
        else:  # "->"
            parent, child = parent_upos, child_upos

        # Пускаем поиск
        results = finder.find_in_directory(
            "./conllu_DeepPavlov",
            child_upos=child,
            child_deprel=deprel,
            parent_upos=parent,
            filename_mask=mask,
        )
        amount = [len(items) for items in results.values()]
        amount = sum(amount)            
        print(amount)

        # Берем первый файл и первый пример
        for filename, items in results.items():
            if not items:
                continue
            item = items[0]
            print(f"Файл: {filename}")
            print(f'"{item["text"]}"')
            print("MATCH:", item["match"])
            print("PARENT:", item["parent"])
            print("-" * 50)

            example_found = True
            break  # один пример на признак

        if example_found:
            break  # выходим из маски, уже нашли пример

    if not example_found:
        print("Не найдено ни одного примера для этого признака.")


=== VERB, <-, PART, parataxis ===
21
Файл: conllu_DeepPavlov\Mamina_love stories.txt.conllu
"— Ну, а раз так, — обрадовался Кузьмич и подмигнул мне (чего, кстати, с ним почти никогда не случалось), — поехали ко мне на Кубань."
MATCH: {'id': 9, 'form': 'обрадовался', 'lemma': 'обрадоваться', 'upos': 'VERB', 'head': 2, 'deprel': 'parataxis'}
PARENT: {'id': 2, 'form': 'Ну', 'lemma': 'ну', 'upos': 'PART', 'head': 28, 'deprel': 'parataxis'}
--------------------------------------------------

=== PRON, <-, ADP, fixed ===
4
Файл: conllu_DeepPavlov\Mamina_love stories.txt.conllu
"Ты на работе и прежде всего должна быть профессионалом."
MATCH: {'id': 6, 'form': 'всего', 'lemma': 'всего', 'upos': 'PRON', 'head': 5, 'deprel': 'fixed'}
PARENT: {'id': 5, 'form': 'прежде', 'lemma': 'прежде', 'upos': 'ADP', 'head': 7, 'deprel': 'advmod'}
--------------------------------------------------

=== PRON, ->, NOUN, nsubj ===
30
Файл: conllu_DeepPavlov\Mamina_love stories.txt.conllu
"Этот проклятый ступор в